# Joint, Marginal & Conditional Distributions

## What's covered

- **Joint distributions** — describing two (or more) random variables at once
- **Marginal distributions** — recovering single-variable distributions from a joint
- **Conditional distributions** — `p(y | x)` and what it means in continuous and discrete cases
- **Independence** and **conditional independence** — restated in distributional language
- The **chain rule of probability** — every joint factors as a product of conditionals
- **Conditional expectation** `\mathbb{E}[Y | X]` and the **tower property**
- Where this appears in ML — probabilistic models, graphical models, supervised learning itself


## Joint distributions

So far we've worked with one random variable at a time. Real ML problems are about *many* random variables — features, labels, latents — and how they interact. The **joint distribution** of `X` and `Y` describes the probability of each *combination* of values.

**Discrete joint PMF.**

$$
p_{X, Y}(x, y) = P(X = x, Y = y)
$$

A table — one entry per combination, all non-negative, summing to 1.

**Continuous joint PDF.**

$$
f_{X, Y}(x, y) \quad \text{such that} \quad P((X, Y) \in A) = \iint_A f_{X, Y}(x, y) \, dx \, dy
$$

The joint is a non-negative function that integrates to 1 over the plane.

**For more than two.** Same idea: `p(x_1, x_2, \dots, x_d)` describes the joint over `d` random variables. In ML this is *the* object of interest — `p(\text{features, label})`, `p(\text{input, latent, output})`, `p(\text{token}_1, \text{token}_2, \dots, \text{token}_n)`.

**Why joints are hard.** A joint over `d` binary variables has `2^d - 1` independent parameters. For `d = 30` that's already over a billion. The whole story of probabilistic ML is *exploiting structure* — independence, low rank, factorization — to make joint distributions tractable.


## Marginal distributions — sum (or integrate) out

If you have the joint `p(x, y)` and you only care about `X`, you **marginalize out** `Y`:

$$
p_X(x) = \sum_y p_{X, Y}(x, y) \quad \text{(discrete)}
$$
$$
f_X(x) = \int f_{X, Y}(x, y) \, dy \quad \text{(continuous)}
$$

In words: sum (or integrate) the joint over all values of `Y`. What you get is the distribution of `X` *averaged over* `Y`'s uncertainty.

Geometrically: collapse the joint table by summing rows or columns; collapse the joint surface by integrating along one axis.

**Marginalization in the wild.** "The probability that an email is spam, regardless of who sent it" is a marginal over the sender variable. "The probability of next token being `the`, regardless of what comes after" is a marginal over later tokens.

In Bayesian inference, the **evidence** `p(D) = \int p(D | \theta) p(\theta) \, d\theta` is a marginal — we sum out the parameters to get the data's "prior predictive" likelihood. This integral is the thing that makes Bayesian inference hard, and it's why we invent variational and Monte Carlo methods.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(0)

# Joint PMF for a small discrete example: X = die value, Y = is_even
# Build the joint table and check that marginalization recovers correct marginals
# Outcomes of one fair die roll
joint = np.zeros((6, 2))
for x in range(1, 7):
    y = x % 2 == 0
    joint[x - 1, int(y)] = 1/6   # each (x, y) combo with prob 1/6

print("Joint p(X = x, Y = even):")
print(f"{'X':>3} {'odd':>10} {'even':>10}")
for i, x in enumerate(range(1, 7)):
    print(f"{x:>3} {joint[i, 0]:>10.4f} {joint[i, 1]:>10.4f}")
print(f"\nTotal probability sum: {joint.sum():.4f}  (should be 1)\n")

# Marginalize: sum out Y → p_X(x); sum out X → p_Y(y)
p_X = joint.sum(axis=1)
p_Y = joint.sum(axis=0)

print(f"p_X(x) by marginalizing out Y: {p_X.round(4)}")
print(f"p_Y(y = odd):  {p_Y[0]:.4f}")
print(f"p_Y(y = even): {p_Y[1]:.4f}")


## Conditional distributions

Given the joint, the **conditional** distribution of `Y` given `X = x` is

$$
p_{Y | X}(y | x) = \frac{p_{X, Y}(x, y)}{p_X(x)} \quad \text{(discrete)}
$$
$$
f_{Y | X}(y | x) = \frac{f_{X, Y}(x, y)}{f_X(x)} \quad \text{(continuous)}
$$

In words: fix `X = x`, restrict to the slice of the joint corresponding to that fixed value, and renormalize. The result is a *real* probability distribution over `y` — sums to 1 (or integrates to 1) for any fixed `x`.

**Geometric picture.** For a 2D joint density `f_{X, Y}(x, y)`, slicing along a horizontal line `Y = y` gives a curve as a function of `x`. That curve, divided by its area, *is* the conditional density of `X` given `Y = y`.

**The three central formulas of probability**, in compact form:

$$
\underbrace{p(x, y)}_{\text{joint}} = \underbrace{p(y | x)}_{\text{conditional}} \cdot \underbrace{p(x)}_{\text{marginal}}
$$

Memorize this. From it you can derive Bayes' theorem, marginalization, independence — everything.

**ML angle.** Every supervised learning model outputs a conditional distribution: classifiers output `p(y | x)` (categorical), regressors output `p(y | x)` (typically Gaussian centered on the prediction). When a paper says "a probabilistic model of inputs and outputs," they mean a joint `p(x, y)`. When they say "a discriminative model," they mean *just* the conditional `p(y | x)`. Generative models give you both.


In [ ]:
# Continuous conditional example: two correlated Gaussians
# Set up X, Y ~ jointly Gaussian with cov(X, Y) = 0.7
N = 200_000
mean = [0, 0]
cov = [[1.0, 0.7], [0.7, 1.0]]
joint = rng.multivariate_normal(mean, cov, size=N)
X, Y = joint[:, 0], joint[:, 1]

# Conditional distribution of Y given X = 1: by theory, Y | X = x ~ N(0.7*x, 1 - 0.7^2) = N(0.7, 0.51)
# Empirically: take samples where X is close to 1
near = np.abs(X - 1.0) < 0.05
Y_cond = Y[near]
print(f"Empirical Y | X ≈ 1:  mean = {Y_cond.mean():.4f}  (analytic 0.7)")
print(f"                       var  = {Y_cond.var(ddof=1):.4f}  (analytic 0.51)")

# Now condition on a different value: X ≈ -2
near = np.abs(X + 2.0) < 0.05
Y_cond = Y[near]
print(f"\nEmpirical Y | X ≈ -2: mean = {Y_cond.mean():.4f}  (analytic -1.4)")
print(f"                       var  = {Y_cond.var(ddof=1):.4f}  (analytic 0.51)")

print("\nNotice: the conditional MEAN shifts with the conditioning value,")
print("but the conditional VARIANCE stays the same (Gaussian special property).")


## Independence — in distributional language

We saw event-level independence in notebook 1. For random variables, the same idea takes a clean factorization form:

$$
X \perp\!\!\!\perp Y \quad \Longleftrightarrow \quad p(x, y) = p(x) \, p(y) \text{ for all } x, y
$$

Equivalently: the conditional equals the marginal — `p(y | x) = p(y)` for all `x`. Knowing `X` tells you nothing about `Y`.

**Conditional independence** — `X` and `Y` are conditionally independent given `Z` if

$$
p(x, y | z) = p(x | z) \, p(y | z) \text{ for all } x, y, z \quad \Longleftrightarrow \quad X \perp\!\!\!\perp Y \mid Z
$$

In ML this is the more useful notion. Two pixels in an image are not independent — they're correlated through the underlying scene. But *given the latent representation* of that scene, they often become approximately independent. That's the modeling idea behind autoencoders and other latent-variable models.

**Why factorization matters in practice.** A general joint over `d` variables has `O(\exp(d))` parameters. But if it factorizes as `p(x_1) p(x_2 | x_1) p(x_3 | x_2)` (a Markov chain), it has only `O(d)` parameters. **Independence structure is the primary tool for taming high dimensions in probabilistic ML.**


## The chain rule of probability

Every joint distribution factors into a product of conditionals — for any ordering:

$$
p(x_1, x_2, \dots, x_n) = p(x_1) \, p(x_2 | x_1) \, p(x_3 | x_1, x_2) \, \cdots \, p(x_n | x_1, \dots, x_{n-1})
$$

This is the **chain rule of probability**. It's exact — no approximation — and it holds for *any* ordering of the variables. Pick the ordering most convenient for your model.

**For ML:** this is the entire story of autoregressive modeling.

- **Language models.** `p(\text{tokens}) = \prod_t p(t_t | t_1, \dots, t_{t-1})`. GPT trains by maximizing this product — also known as "next-token prediction."
- **Autoregressive image models.** Pixel-by-pixel, raster order. PixelRNN, PixelCNN.
- **Autoregressive audio.** WaveNet predicts each sample given the past.
- **Bayesian networks.** A directed graph specifies a chain-rule factorization where each conditional only depends on the *parents* in the graph — exploiting conditional independence to shrink the parameter count.

The chain rule is also what lets us derive Bayes' theorem: write `p(x, y)` two ways using the chain rule with `x` first vs `y` first, then equate.


## Conditional expectation

The **conditional expectation** of `Y` given `X = x` is the expectation under the conditional distribution:

$$
\mathbb{E}[Y | X = x] = \sum_y y \, p_{Y | X}(y | x) \quad \text{(discrete)}
$$
$$
\mathbb{E}[Y | X = x] = \int y \, f_{Y | X}(y | x) \, dy \quad \text{(continuous)}
$$

Two readings:

- For a fixed value `x`, `\mathbb{E}[Y | X = x]` is a **number** — the average `Y` over the slice of the joint where `X = x`.
- As `x` varies, `\mathbb{E}[Y | X = \cdot]` becomes a **function of `x`** — often written `\mathbb{E}[Y | X]` (note: no specific value). This *function* is itself a random variable, because it's a function of the random variable `X`.

**ML angle.** The optimal regression function under squared loss is exactly the conditional expectation:

$$
\arg\min_g \mathbb{E}[(Y - g(X))^2] = g^*(x) = \mathbb{E}[Y | X = x]
$$

This is why MSE regression is fitting a model of the conditional mean. Mean prediction, not the whole distribution — which is why MSE doesn't capture aleatoric uncertainty.

Similarly, the optimal classification function under cross-entropy loss is the **conditional class probability** `\mathbb{E}[Y | X = x] = P(Y = 1 | X = x)` (for binary). So classifiers approximate conditional class probabilities.


In [ ]:
# Numerically: optimal regressor for Y | X is E[Y | X]
N = 100_000

# Y = sin(X) + Gaussian noise, X uniform
X = rng.uniform(-3, 3, N)
Y = np.sin(X) + 0.3 * rng.normal(size=N)

# Estimate E[Y | X] by binning X and averaging Y in each bin
bins = np.linspace(-3, 3, 31)
bin_idx = np.digitize(X, bins)
E_Y_given_X = np.array([Y[bin_idx == i].mean() for i in range(1, len(bins))])
centers = 0.5 * (bins[:-1] + bins[1:])

# Compare to the true conditional mean — should be sin(x)
print(f"{'x':>6} {'E[Y|X] estimated':>18} {'sin(x) true':>14}")
for i, x in enumerate(centers[::4]):
    print(f"{x:>6.2f} {E_Y_given_X[i*4]:>18.4f} {np.sin(x):>14.4f}")
print("\nThe estimated E[Y|X] tracks sin(x) — confirming the optimal regressor under MSE.")


## The tower property — `E[E[Y | X]] = E[Y]`

A simple but extraordinarily useful identity:

$$
\mathbb{E}\bigl[\mathbb{E}[Y | X]\bigr] = \mathbb{E}[Y]
$$

(Also called the **law of total expectation** or *iterated expectation*.) In words: averaging the conditional means over `X` gives back the marginal mean of `Y`. If you compute `\mathbb{E}[Y | X = x]` for every `x`, and then average those over the distribution of `X`, you recover `\mathbb{E}[Y]`.

**Why it's useful.**

- **Conditioning can be easier than unconditional.** Some expectations are hard to compute directly but easy once you condition on the right variable. Compute `\mathbb{E}[Y | X]` in closed form, then average over `X`.
- **Variance decomposition.** A similar identity for variance:
  $$
  \text{Var}(Y) = \mathbb{E}[\text{Var}(Y | X)] + \text{Var}(\mathbb{E}[Y | X])
  $$
  This is the **law of total variance**, and it splits total variance into "noise within groups" plus "spread between groups." It's the math behind ANOVA, hierarchical models, and the bias-variance decomposition.
- **Monte Carlo variance reduction.** **Rao-Blackwellization** uses the tower property: replace your noisy estimator `g(X, Y)` with `\mathbb{E}[g(X, Y) | X]` and you get the same expectation but lower (or equal) variance. Powerful tool in particle filtering and REINFORCE.
- **EM algorithm.** The E-step computes `\mathbb{E}[\text{complete-data log-likelihood} | \text{observed data}]`. This conditional expectation is then maximized in the M-step.

**ML angle.** "Conditional on the dataset, the parameter estimate is ..." → average those out over hypothetical datasets → unconditional bias and variance estimates of the estimator. The tower property is the mechanic that turns conditional analyses into unconditional ones.


## Where this appears in ML

Joint, marginal, and conditional distributions are the structural backbone of *every* probabilistic ML model.

- **Supervised learning.** Trying to learn `p(y | x)` — a conditional. Discriminative models stop there. Generative models also model `p(x, y)` jointly, allowing tasks like missing-data imputation or sample generation.
- **Bayes' theorem in classification.** `p(y | x) = p(x | y) p(y) / p(x)` — a conditional, derived from joint factorization, with marginalization in the denominator.
- **Naive Bayes.** Assumes `p(x_1, \dots, x_d | y) = \prod_i p(x_i | y)` — conditional independence given the class. Cheap, fast, often surprisingly competitive.
- **Graphical models.** Bayesian networks (directed) and Markov random fields (undirected) encode conditional independence structure visually. The joint factorizes according to the graph; inference algorithms exploit the factorization.
- **HMMs.** Hidden Markov models — autoregressive on latent states. Forward-backward algorithm computes marginals; Viterbi computes the most likely state sequence.
- **Variational inference.** Approximate intractable posteriors `p(z | x)` with tractable `q(z | x)`. The KL divergence between them involves both joint and conditional structure.
- **VAEs.** Maximize `p(x) = \int p(x, z) dz` (a marginal over latents). Direct integration is intractable, so we optimize the **evidence lower bound** instead.
- **Diffusion models.** The reverse process is a chain `p(x_{t-1} | x_t)` of conditional Gaussians — autoregressive over time, then chain-ruled to compute `p(x_0)`.
- **Language models.** `p(\text{tokens}) = \prod_t p(t_t | t_{<t})` — chain rule, learned with neural nets.
- **Attention.** Soft conditioning — the attention weights compute a softmax of similarities, then aggregate values, which is structurally `\mathbb{E}_{p(j | i)}[v_j]`.
- **Conditional generative models.** Conditional GANs, conditional diffusion, image-to-image models — explicitly factor `p(\text{output} | \text{input}, \text{noise})`.
- **Causal inference.** Distinguishes correlation (`p(y | x)`) from causation (`p(y | \text{do}(x))`) by reasoning about which conditional independencies survive interventions.

Next notebook: **the Gaussian distribution** — the single most-used distribution in ML, in both its scalar and multivariate forms. We finally do conditioning and marginalization for Gaussians, which give surprisingly clean closed forms.
